# Test-time compute

Wrap any provider or local generator in a larger inference algorithm. Count every call as part of the budget. History and current practice: see [`docs/test-time-compute.md`](../docs/test-time-compute.md).

In [ ]:
from llm_textgen_lab.orchestration import agreement_rate, extract_answer, majority_vote

# The vote is over extracted final answers, which is what self-consistency actually does.
candidates = ['Two plus two is four.\nAnswer: 4', 'I count 2 + 2 = 4.\nAnswer: 4', 'Answer: 5']
majority_vote(candidates, extract=extract_answer), agreement_rate(candidates, extract=extract_answer)

The importable orchestration module also provides self-consistency, draft–critic–revision, verifier selection, best-of-N, a refine loop, and a budget cascade. Use a fake generator in tests first, then plug in an API adapter or Hugging Face generator.

In [ ]:
from llm_textgen_lab.orchestration import best_of_n
from llm_textgen_lab.types import GenerationConfig, GenerationResult

answers = iter(['12', '8', '10'])

def fake(prompt, config):
    return GenerationResult(text=next(answers), provider='fake')

record = best_of_n(fake, 'Reach 8.', GenerationConfig(), score=lambda text: -abs(8 - int(text)), count=3)
record.result.text, record.calls

In [ ]:
from llm_textgen_lab.orchestration import refine_loop

def fixer(prompt, config):
    return GenerationResult(text='revised: tests pass' if 'Revise' in prompt else 'draft: tests fail', provider='fake')

record = refine_loop(fixer, 'Write the function.', GenerationConfig(), accept=lambda text: 'tests pass' in text, max_rounds=2)
record.result.text, record.calls, record.budget_label

In [ ]:
from llm_textgen_lab.orchestration import escalate

def tiered(prompt, config):
    return GenerationResult(text='8' if config.reasoning_effort == 'high' else '7', provider='fake')

configs = [GenerationConfig(reasoning_effort='low'), GenerationConfig(reasoning_effort='high')]
record = escalate(tiered, 'Reach 8.', configs, accept=lambda text: text == '8')
record.result.text, record.calls, record.budget_label

In [ ]:
# Conceptual experiment matrix:
# 1 high-effort call
# 5 low-effort calls + majority vote
# N candidates + programmatic scorer (best-of-n)
# draft + critic + revision, or refine until tests pass (refine-loop)
# candidate generation + verifier
# cheap config first, escalate on failure or disagreement (escalate)
# Compare quality, latency, total tokens, and number of calls.